In [3]:
# =========================================
# TASK 9: SQL DATA MODELING - STAR SCHEMA
# GOOGLE COLAB + SQLITE (SINGLE CODE)
# =========================================

import pandas as pd
import sqlite3

# ---------- LOAD DATASET ----------
df = pd.read_csv("Global_Superstore(CSV).csv")

# ---------- CREATE SQLITE DATABASE ----------
conn = sqlite3.connect("star_schema.db")
cursor = conn.cursor()

# ---------- CREATE DIMENSION TABLES ----------
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT,
    segment TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_product (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT,
    category TEXT,
    sub_category TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_date TEXT,
    year INTEGER,
    month INTEGER,
    day INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_region (
    region_id INTEGER PRIMARY KEY AUTOINCREMENT,
    region TEXT,
    country TEXT
)
""")

# ---------- CREATE FACT TABLE ----------
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_sales (
    sales_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    product_id INTEGER,
    date_id INTEGER,
    region_id INTEGER,
    sales REAL,
    quantity INTEGER,
    profit REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY (region_id) REFERENCES dim_region(region_id)
)
""")

# ---------- INSERT INTO DIMENSIONS ----------
df[['Customer Name','Segment']].drop_duplicates().rename(
    columns={'Customer Name':'customer_name'}
).to_sql('dim_customer', conn, if_exists='append', index=False)

df[['Product Name','Category','Sub-Category']].drop_duplicates().rename(
    columns={'Product Name':'product_name','Sub-Category':'sub_category'}
).to_sql('dim_product', conn, if_exists='append', index=False)

df['Order Date'] = pd.to_datetime(df['Order Date'])
date_df = df[['Order Date']].drop_duplicates()
date_df['year'] = date_df['Order Date'].dt.year
date_df['month'] = date_df['Order Date'].dt.month
date_df['day'] = date_df['Order Date'].dt.day
date_df.rename(columns={'Order Date':'order_date'}).to_sql(
    'dim_date', conn, if_exists='append', index=False
)

df[['Region','Country']].drop_duplicates().rename(
    columns={'Region':'region','Country':'country'}
).to_sql('dim_region', conn, if_exists='append', index=False)

# ---------- LOAD RAW DATA INTO SQLITE ----------
df.to_sql("sales_raw", conn, if_exists='replace', index=False)

# ---------- INSERT INTO FACT TABLE ----------
cursor.execute("""
INSERT INTO fact_sales (
    customer_id, product_id, date_id, region_id,
    sales, quantity, profit
)
SELECT
    c.customer_id,
    p.product_id,
    d.date_id,
    r.region_id,
    s.Sales,
    s.Quantity,
    s.Profit
FROM sales_raw s
JOIN dim_customer c ON s."Customer Name" = c.customer_name
JOIN dim_product p ON s."Product Name" = p.product_name
JOIN dim_date d ON s."Order Date" = d.order_date
JOIN dim_region r ON s.Region = r.region
""")

conn.commit()

# ---------- CREATE INDEXES ----------
cursor.execute("CREATE INDEX IF NOT EXISTS idx_customer ON fact_sales(customer_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_product ON fact_sales(product_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON fact_sales(date_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_region ON fact_sales(region_id)")

# ---------- ANALYTICS QUERY ----------
query = """
SELECT r.region, SUM(f.sales) AS total_sales
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region
"""
analysis = pd.read_sql(query, conn)

# ---------- EXPORT OUTPUT ----------
analysis.to_csv("analysis_outputs.csv", index=False)

# ---------- SAVE SQL FILE ----------
sql_file = """
-- TASK 9: STAR SCHEMA SQL
-- Dimension Tables + Fact Table
-- Google Colab + SQLite
"""

with open("task9_star_schema.sql", "w") as f:
    f.write(sql_file)

print(" TASK 9 COMPLETED SUCCESSFULLY")
print("Files generated:")
print("1. star_schema.db")
print("2. analysis_outputs.csv")
print("3. task9_star_schema.sql")


 TASK 9 COMPLETED SUCCESSFULLY
Files generated:
1. star_schema.db
2. analysis_outputs.csv
3. task9_star_schema.sql
